## Data Preparation

In [ ]:
def read_documents(doc_file, label_col=1):
    """label_col=1: polarity (pos/neg), label_col=0: topic."""
    docs, labels = [], []
    with open(doc_file, encoding='utf-8') as f:
        for line in f:
            words = line.strip().split()
            docs.append(words[3:])
            labels.append(words[label_col])
    return docs, labels

def split(docs, labels, frac=0.80):
    sp = int(frac * len(docs))
    return docs[:sp], labels[:sp], docs[sp:], labels[sp:]

## Data Training

In [ ]:

from collections import Counter
import math

def train_nb(documents, labels, k=1.0):
    label_counts = Counter(labels)
    word_counts = {c: Counter() for c in label_counts} # count of words for each label
    for doc, c in zip(documents, labels):
        word_counts[c].update(doc)
    vocab = set(w for c in word_counts for w in word_counts[c])
    total_words = {c: sum(word_counts[c].values()) for c in label_counts}
    n = len(documents)
    log_label_prob = {c: math.log(label_counts[c] / n) for c in label_counts} # how often each label occurs
    return log_label_prob, word_counts, total_words, vocab, k

In [ ]:
def score_doc_label(document, label, model):
    log_label_prob, word_counts, total_words, vocab, k = model
    score = log_label_prob[label]
    denom = total_words[label] + k * len(vocab)
    for w in document:
        if w not in vocab:  # skip unseen words
            continue
        score += math.log((word_counts[label][w] + k) / denom)
    return score